# **HW10 – Quantum CNNs, GNNs, Bayes, and K-Means**
_Time required: ~2–3 hours (for students with Qiskit ML experience)_

**What you’ll practice**
- Building quanvolutional layers for QCNNs
- Encoding graphs and message passing in QGNNs
- Quantum Bayesian inference with phase estimation
- Quantum distance estimation for K-means clustering
- Hybrid implementations and evaluations
- Noise impact and mitigations

**What to turn in**
- This single notebook (`HW10_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later), **Qiskit Machine Learning**.
- Use AerSimulator for reproducibility.
- If stuck, explain reasoning; partial credit for clear work.
- Use MNIST digits (small 4x4) for QCNN; MUTAG graphs (small subset) for GNN; toy probs for Bayes/K-means.
- Focus on small scales: 2-4 qubits, reps=1.


In [ ]:
# --- Setup (run me first) ---
# Install if needed (uncomment in Colab)
# !pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-machine-learning matplotlib sklearn networkx

# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_algorithms.optimizers import SPSA
from sklearn import datasets, metrics
from sklearn.cluster import KMeans
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Simulator backend
sim = AerSimulator()

# Function to run circuit and get counts
def get_counts(circ, shots=1024):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-4):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close: {A} vs {B}")


## Part A — Quantum CNNs (Session 14) (≈30 min)

**A1.** Implement a 2x2 quanvolutional filter (2 qubits, ry+cz+ry) on a 4x4 MNIST digit patch. Compute <Z> output.  
**A2.** Apply to full digit; compare feature map size to classical conv (kernel=2).  
**A3.** Short answer: Parameter sharing in quanv; pooling methods.


In [ ]:
# A1. Quanv filter
digits = datasets.load_digits()
img = digits.images[0][:4, :4].flatten() / 16  # Normalize patch
qc_quanv = QuantumCircuit(2)
# YOUR CODE HERE: ry(img[i]*π, i) for encoding; cz(0,1); ry(params); measure z(0)
qc_quanv.draw('mpl')
plt.show()
z_exp = ...  # Compute <Z>
print("<Z>: ", z_exp)

# A2. Full feature map
# Stride-1 on 4x4 → 3x3 features; YOUR CODE HERE: loop patches
features = ...
print("Quanv features shape: ", features.shape)

# A3. Written answer: (Same circuit/params per patch; measure-reduce or unitary entangle+discard)


## Part B — Quantum GNNs (Session 15) (≈45 min)

**B1.** Encode a small graph (cycle-4) with node features as angles; draw ego-graph for node 0 (radius=1).  
**B2.** Implement quantum message passing (2 qubits, ry+cx+ry) on ego; aggregate with mean.  
**B3.** Short answer: Ego-graph scalability; quantum vs classical message passing.


In [ ]:
# B1. Graph encode
G = nx.cycle_graph(4)
nx.set_node_attributes(G, {i: np.random.rand(2) for i in G.nodes}, 'feat')
ego_0 = nx.ego_graph(G, 0, radius=1)
nx.draw(ego_0, with_labels=True)
plt.show()

# B2. Message passing
qc_mp = QuantumCircuit(2)
# YOUR CODE HERE: encode node+neigh feats; cx; ry param; <Z> out
agg = np.mean([<Z> for neigh in ego_0.neighbors(0)] + [<Z_self>])
print("Aggregated: ", agg)

# B3. Written answer: (Fixed qubits per ego; quantum entangles messages, classical linear)


## Part C — Quantum Bayes (Session 16) (≈45 min)

**C1.** Encode prior p=0.3 as ry(θ)|0>; update with likelihood 0.8 via controlled-ry.  
**C2.** Estimate posterior with shots=512; compare to classical Bayes.  
**C3.** Short answer: Phase estimation for probs; quantum nets vs classical.


In [ ]:
# C1. Prior/likelihood
theta_p = 2 * np.arcsin(np.sqrt(0.3))
qc_bayes = QuantumCircuit(2, 1)
# YOUR CODE HERE: ry(theta_p,0); x(0) for control-on-0 if needed; cry for lik
qc_bayes.draw('mpl')
plt.show()

# C2. Posterior est
qc_bayes.measure(1,0)
counts = get_counts(qc_bayes)
post = counts.get('1',0) / sum(counts.values())
classical = (0.8 * 0.3) / (0.8*0.3 + 0.2*0.7)  # Bayes with false pos 0.2
print("Quantum post: ", post, " Classical: ", classical)

# C3. Written answer: (QPE for amp → prob; quantum handles superposition conditionals)


## Part D — Quantum K-Means (Session 17) (≈30 min)

**D1.** Implement swap-test distance for two iris points (normalize vectors).  
**D2.** Run hybrid K-means (k=2) on iris (2 feats); compare silhouette to classical.  
**D3.** Short answer: Amplitude encoding advantage; hybrid loop.


In [ ]:
# D1. Swap distance
iris = datasets.load_iris()
x1 = iris.data[0, :2] / np.linalg.norm(iris.data[0, :2])
x2 = iris.data[50, :2] / np.linalg.norm(iris.data[50, :2])
qc_swap = QuantumCircuit(3,1)
# YOUR CODE HERE: amp encode x1 on 1, x2 on 2; swap test
fid = ...
dist = np.sqrt(2 - 2 * fid)
print("Dist: ", dist)

# D2. Hybrid K-means
X = iris.data[:, :2]
# YOUR CODE HERE: quantum dist matrix; classical assign/update
sil_q = metrics.silhouette_score(X, labels_q)
kmeans_c = KMeans(n_clusters=2).fit(X)
sil_c = metrics.silhouette_score(X, kmeans_c.labels_)
print("Q sil: ", sil_q, " C sil: ", sil_c)

# D3. Written answer: (Log-dim qubits; quantum dists, classical centroids/assigns)


## Part E — Noise & Applications (≈30 min)

**E1.** Add noise to swap-test; recompute dist for A1 points.  
**E2.** Short answer: SQD in Bayes; quantum advantage regimes.  
**E3.** Written reflection: Apps in vision (QCNN), graphs (QGNN), UQ (Bayes), clustering (K-means).


In [ ]:
# E1. Noisy dist
from qiskit_aer.noise import NoiseModel, depolarizing_error
noise_model = NoiseModel()
# YOUR CODE HERE: add depol 0.01 to 'cx','h'
counts_noisy = get_counts(qc_swap, noise_model=noise_model)
fid_noisy = ...
dist_noisy = np.sqrt(2 - 2 * fid_noisy)
print("Noisy dist: ", dist_noisy)

# E2. Written answer: (Stochastic channels for robust UQ; high-dim entangled probs/clusters)

# E3. Written answer: (QCNN image feats; QGNN mol/social; Bayes AI uncertainty; K-means quantum data grouping)
